# Applications

Real-world applications of Hidden Markov Models. These applications leverage the three fundamental HMM operations:

- **Forward algorithm**: Compute $P(O|\lambda)$ for classification and anomaly detection
- **Viterbi algorithm**: Find optimal state sequence for sequence labeling
- **Baum-Welch algorithm**: Learn model parameters from training data

In [ ]:
import numpy as np

from hmm import HMM, HMMClassifier, baum_welch, forward, viterbi

## 1. Binary Classification

Use two HMMs to classify observations as positive or negative.

In [ ]:
# Train HMMs for two classes
V = [0, 1, 2]

# Class 0: prefers symbols 0, 1
A0 = np.array([[0.7, 0.3], [0.3, 0.7]])
B0 = np.array([[0.6, 0.3, 0.1], [0.5, 0.3, 0.2]])
hmm_class0 = HMM(n_states=2, A=A0, B=B0, V=V)

# Class 1: prefers symbols 1, 2
A1 = np.array([[0.7, 0.3], [0.3, 0.7]])
B1 = np.array([[0.1, 0.3, 0.6], [0.2, 0.3, 0.5]])
hmm_class1 = HMM(n_states=2, A=A1, B=B1, V=V)

# Create classifier
classifier = HMMClassifier(pos_hmm=hmm_class1, neg_hmm=hmm_class0)

# Test observations
test_obs_0 = [0, 0, 1, 0, 1]  # More likely from class 0
test_obs_1 = [2, 2, 1, 2, 1]  # More likely from class 1

score_0 = classifier.classify(test_obs_0)
score_1 = classifier.classify(test_obs_1)

print(f"Observation: {test_obs_0}")
print(f"Classification score: {score_0:.4f} (negative = class 0)\n")

print(f"Observation: {test_obs_1}")
print(f"Classification score: {score_1:.4f} (positive = class 1)")

## 2. Sequence Generation

Generate observation sequences from an HMM.

In [ ]:
# Simple sequence generation
def generate_sequence(hmm, length):
    """Generate observation sequence from HMM."""
    # Start from initial state
    state = np.random.choice(hmm.N, p=hmm.Pi)
    obs = []

    for _ in range(length):
        # Emit observation
        obs.append(np.random.choice(hmm.M, p=hmm.B[state]))
        # Transition to next state
        state = np.random.choice(hmm.N, p=hmm.A[state])

    return obs

# Generate sequences
V = [0, 1, 2]
A = np.array([[0.8, 0.2], [0.2, 0.8]])
B = np.array([[0.7, 0.2, 0.1], [0.1, 0.2, 0.7]])
hmm = HMM(n_states=2, A=A, B=B, V=V)

print("Generated sequences:")
for i in range(5):
    seq = generate_sequence(hmm, 10)
    print(f"  {seq}")

## 3. Anomaly Detection

Use forward probability to detect unusual sequences.

In [ ]:
# Train on normal sequences
normal_seqs = [
    [0, 0, 0, 1, 1, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 0, 0, 1, 1, 0, 0],
]

V = [0, 1]
hmm_normal = HMM(n_states=2, V=V)
hmm_normal = baum_welch(hmm_normal, obs_seqs=normal_seqs, epochs=20)

# Test sequences
normal_test = [0, 0, 1, 1, 0, 0, 1, 1]
anomaly_test = [1, 1, 1, 1, 1, 1, 1, 1]

normal_ll = forward(hmm_normal, normal_test, scaling=True)[0]
anomaly_ll = forward(hmm_normal, anomaly_test, scaling=True)[0]

print(f"Normal sequence log-likelihood: {normal_ll:.4f}")
print(f"Anomaly sequence log-likelihood: {anomaly_ll:.4f}")
print("\nLower likelihood = more unusual (anomalous)")

## 4. Part-of-Speech Tagging

Use HMM to infer hidden POS tags from observed words.

In [ ]:
# Simple POS tagging example
# States: N (noun), V (verb), D (determiner)
# Observations: words

# Map words to indices
word_to_idx = {'the': 0, 'cat': 1, 'dog': 2, 'runs': 3, 'eats': 4}
idx_to_word = {v: k for k, v in word_to_idx.items()}
V = list(word_to_idx.values())

# Simple HMM for POS
A = np.array([  # N->N, N->V, V->N, V->D
    [0.3, 0.3, 0.4],  # From N: N, V, D
    [0.4, 0.3, 0.3],  # From V: N, V, D
    [0.4, 0.3, 0.3],  # From D: N, V, D
])

# Emission: P(word | tag)
# Tag 0 = Noun, Tag 1 = Verb, Tag 2 = Determiner
B = np.array([
    [0.1, 0.4, 0.4, 0.05, 0.05],  # Noun: cat, dog
    [0.05, 0.05, 0.05, 0.45, 0.45],  # Verb: runs, eats
    [0.5, 0.0, 0.0, 0.0, 0.0],  # Determiner: the
])

hmm_pos = HMM(n_states=3, A=A, B=B, V=V)

# Tag a sentence
sentence = ['the', 'cat', 'runs']
obs = [word_to_idx[w] for w in sentence]

states, _, _ = viterbi(hmm_pos, obs, scaling=True)
pos_tags = ['Noun', 'Verb', 'Determiner']

print("Sentence:", sentence)
print("Predicted POS:", [pos_tags[s] for s in states])

## Summary

HMMs are used for:
- **Classification**: Binary/multi-class with multiple HMMs
- **Generation**: Sample sequences from learned models
- **Anomaly Detection**: Identify unusual sequences
- **Sequence Labeling**: POS tagging, gene finding, etc.

This concludes the HMMPY tutorial. For more information, see the [documentation](https://bio-comp.github.io/hmmpy/).